In [1]:
%matplotlib ipympl
import pandas as pd
import numpy as np
# from scipy.stats import kruskal, mannwhitneyu, ttest_rel, wilcoxon
# import itertools
import statistic_functions as stats
from scipy.io import loadmat
import matplotlib.pyplot as plt
import os 

from importlib import reload
reload(stats)

<module 'statistic_functions' from 'c:\\Users\\aless\\Desktop\\Riemann-centroids-study\\statistic_functions.py'>

In [2]:
f1_1_winterStopX = 75
f1_2_bhbf = 79
g6_2_feetX = 39

parent_dir = os.path.abspath('..')

pathData = os.path.join(parent_dir,'cybathlon_results')
savingPath = f'{pathData}/images'

In [ ]:
user1 = pd.read_csv(f"{pathData}/results_user_2019.csv")
user2 = pd.read_csv(f"{pathData}/results_user_2023.csv")

user1['dayId'] = user1.nDay
user1['dts'] = 0
user1['nSegment'] = 0
user1.loc[user1.nRun>=f1_1_winterStopX,'nSegment'] = 1

user2['dayId'] = user2.nDay
user2['nDay'] += user1.nDay.max() + 1
user2['dts'] = 1
user2['nSegment'] = 2
user2.loc[user2.nRun>=f1_2_bhbf,'nSegment'] = 3
user2['nRun'] += user1.nRun.max() + 1


for clm in user2.columns[user2.columns.str.contains(r'\(')]:
    user1[clm] = user1[clm[1:-1]]

user = pd.concat([user1, user2], ignore_index=True)       

count = -1
dayOld = -1
for idx, row in user.iterrows():
    if row['dayId'] != dayOld:
        count += 1
        dayOld = row['dayId']
    user.at[idx, 'nDay'] = count

for clm in user.columns[user.columns.str.contains('angle')]:
    stats.add_cos_sin_angles(user, clm)


In [4]:
import matplotlib.pyplot as plt
import numpy as np

def polar_plot_by_task_band(
    df,
    radius_col,
    angle_col,
    task_col="task",
    band_col="band",
    segment_col="nSegment",
    r_max=None,
    figsize=(10, 10),
    alpha=0.8,
    s=25,
    cmap_name="tab10"
):
    """
    2x2 polar plot split by task (0/1) and band (0/1),
    colored categorically by nSegment using tab10.
    """

    fig, axes = plt.subplots(
        2, 2,
        subplot_kw={"projection": "polar"},
        figsize=figsize
    )

    axes = np.atleast_2d(axes)

    # --- same logic as plot_df_runs_grid ---
    segments = np.sort(df[segment_col].dropna().unique())
    cmap = plt.get_cmap(cmap_name)
    color_map = {
        seg: cmap(i % cmap.N)
        for i, seg in enumerate(segments)
    }

    for task in [0, 1]:
        for band in [0, 1]:
            ax = axes[task, band]

            subset = df[
                (df[task_col] == task) &
                (df[band_col] == band)
            ]

            for seg in segments:
                seg_data = subset[subset[segment_col] == seg]
                if seg_data.empty:
                    continue

                ax.scatter(
                    seg_data[angle_col],
                    seg_data[radius_col],
                    color=color_map[seg],
                    alpha=alpha,
                    s=s,
                    edgecolors="k",
                    label=f"{segment_col}={seg}"
                )
            ax.set_thetamin(0)
            ax.set_thetamax(180)

            if r_max is not None:
                ax.set_rmax(r_max)

            ax.set_title(f"Task {task}, Band {band}")

    # # --- shared legend (categorical, like your other plot) ---
    # handles, labels = axes[0, 0].get_legend_handles_labels()
    # if handles:
    #     fig.legend(
    #         handles,
    #         labels,
    #         loc="lower center",
    #         ncol=min(len(segments), 5),
    #         frameon=False
    #     )

    fig.suptitle(
        f"Polar plot: {radius_col} vs {angle_col}",
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.show()

    return fig


In [ ]:
saveFigure = False
useuser = True

column_name = 'distance' 
averageOnTask = True
ylim = [-1,3]


if useuser:    
    df = user
    saveName = 'user'
else:   
    pass


fig = stats.plot_df_runs_grid(
    df, column_name,           
    fitColumn='nSegment',           # column name with segment labels (discrete ints/labels). If None -> no segmentation (whole series)
    xlim=None, ylim=ylim,             # axis limits (tuples/lists like [xmin, xmax] or None)
    averageOnTask=averageOnTask,      # average across tasks for each (day, runId, band)
    mergeTask=False,                 # merge tasks (ignore task column)
    fitLine=True,                   # fit line to each segment
    useHuber=True,                   # attempt robust fit 
    title=f"{column_name} - {saveName}",    # figure title
)

if saveFigure:
    saveName = f"{saveName}_{column_name}.svg"
    # if mergeTask:       saveName = saveName.replace('.svg', '_merged.svg')
    if averageOnTask:   saveName = saveName.replace('.svg', '_avg.svg')
    fig.savefig(
        f"{savingPath}/{saveName}",
        dpi=1000,
        bbox_inches='tight',
    )

# PLOTTING

In [ ]:
from scipy.io import savemat
import riemann_utils.matrix_functions as mrtrf
import numpy as np
import matplotlib.pyplot as plt

from pyriemann.utils.distance import distance_riemann
from pyriemann.utils.mean import mean_riemann

from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import MDS


pathDataCentr = '/home/palatella/workspace/cybathlon_user/'

doLogBandPower = False
applyLaplacian = True

saveName = 'run_centroids_user.mat' if not doLogBandPower else 'run_centroids_logBandPower_user.mat'
if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

run_centroids = loadmat(f'{pathDataCentr}{saveName}')
run_centroids2019 = run_centroids['run_centroids']

filename = 'run_centroids_user_20232024.mat'
# filename = 'run_centroids_user_20232024_20192020ref.mat'
if doLogBandPower:
    filename = filename.replace('centroids_user', 'centroids_logBandPower_user')
if not applyLaplacian:
    filename = filename.replace('user', 'user_noLap')

run_centroids = loadmat(f'{pathDataCentr}{filename}')

run_centroids2023 = run_centroids['run_centroids']

run_centroids = np.concatenate((run_centroids2019, run_centroids2023), axis=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyriemann.utils.distance import distance_riemann
from pyriemann.utils.mean import mean_riemann
from pyriemann.utils.tangentspace import log_map_riemann
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import MDS
from scipy.spatial.distance import pdist, squareform
from collections import Counter

# -------------------------
# 1. Core Riemannian utilities
# -------------------------

def evaluate_embedding_quality(representatives, embedded_points, knn_k=5):
    D_true = riemannian_distance_matrix(representatives)
    return {
        "distance_distortion": distance_distortion(D_true, embedded_points),
        "knn_preservation": knn_preservation(D_true, embedded_points, knn_k)
    }

def riemannian_distance_matrix(matrices):
    n = len(matrices)
    d = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            dis = distance_riemann(matrices[i], matrices[j])
            d[i, j] = d[j, i] = dis
    return d

def cluster_means(matrices, labels):
    reps = []
    for lab in np.unique(labels):
        cluster = np.array([matrices[i] for i in range(len(matrices)) if labels[i] == lab])
        reps.append(mean_riemann(cluster))
    return np.array(reps)

def embed_3d(matrices, random_state=0):
    d = riemannian_distance_matrix(matrices)
    x3 = MDS(
        n_components=3,
        dissimilarity='precomputed',
        random_state=random_state
    ).fit_transform(d)
    return x3

# -------------------------
# 2. Adaptive radius estimation
# -------------------------
def estimate_geodesic_radius(distance_matrix, n_points, k_fraction=0.05, percentile=50):
    k = max(1, int(n_points * k_fraction))  # at least 1 neighbor
    knn_distances = []
    for i in range(n_points):
        dists = np.sort(distance_matrix[i])
        dists = dists[dists > 0]  # ignore self-distance
        knn_distances.append(dists[min(k-1, len(dists)-1)])
    return np.percentile(knn_distances, percentile)

def riemannian_clustering_auto(distance_matrix, radius):
    try:
        model = AgglomerativeClustering(
            n_clusters=None,
            metric='precomputed',
            linkage='average',
            distance_threshold=radius
        )
    except TypeError:
        model = AgglomerativeClustering(
            n_clusters=None,
            affinity='precomputed',
            linkage='average',
            distance_threshold=radius
        )
    return model.fit_predict(distance_matrix)

# -------------------------
# 3. Embedding quality metrics
# -------------------------
def distance_distortion(original_distances, embedded_points):
    D_emb = squareform(pdist(embedded_points))
    num = np.linalg.norm(original_distances - D_emb, ord='fro')
    den = np.linalg.norm(original_distances, ord='fro')
    return num / den

def knn_preservation(original_distances, embedded_points, k=5):
    D_emb = squareform(pdist(embedded_points))
    n = original_distances.shape[0]
    score = 0.0
    for i in range(n):
        nn_orig = set(np.argsort(original_distances[i])[1:k+1])
        nn_emb  = set(np.argsort(D_emb[i])[1:k+1])
        score += len(nn_orig & nn_emb) / k
    return score / n

def tangent_angle_distortion(representatives, embedded_points):
    """
    Measures angle distortion in tangent space.
    """
    n = len(representatives)
    angles_original = np.zeros((n, n))
    angles_embedded = np.zeros((n, n))
    
    center = mean_riemann(representatives)
    tan_vecs = [log_map_riemann(rep, center) for rep in representatives]
    
    # Original tangent-space angles
    for i in range(n):
        for j in range(i+1, n):
            num = np.trace(tan_vecs[i] @ tan_vecs[j])
            denom = np.sqrt(np.trace(tan_vecs[i] @ tan_vecs[i]) * np.trace(tan_vecs[j] @ tan_vecs[j]))
            angles_original[i,j] = angles_original[j,i] = num / denom
    
    # 3D embedding angles
    for i in range(n):
        for j in range(i+1, n):
            v1, v2 = embedded_points[i], embedded_points[j]
            cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
            angles_embedded[i,j] = angles_embedded[j,i] = cos_theta
    
    return np.linalg.norm(angles_original - angles_embedded, ord='fro') / np.linalg.norm(angles_original, ord='fro')

def variance_preservation(representatives, embedded_points):
    """
    Fraction of tangent-space variance preserved in embedding.
    """
    center = mean_riemann(representatives)
    tan_vecs = np.array([log_map_riemann(rep, center).flatten() for rep in representatives])
    var_original = np.var(tan_vecs, axis=0).sum()
    var_embedded = np.var(embedded_points, axis=0).sum()
    return var_embedded / var_original

def evaluate_embedding_quality_full(representatives, embedded_points, knn_k=5):
    metrics = evaluate_embedding_quality(representatives, embedded_points, knn_k=knn_k)
    metrics["angle_distortion"] = tangent_angle_distortion(representatives, embedded_points)
    metrics["variance_preservation"] = variance_preservation(representatives, embedded_points)
    return metrics

# -------------------------
# 4. Full adaptive pipeline per band/task
# -------------------------
def process_band_task_auto_adaptive(
    matrices,
    k_fraction=0.05,
    percentile=50,
    random_state=0,
    eval_knn=5
):
    n_points = len(matrices)
    d = riemannian_distance_matrix(matrices)
    radius = estimate_geodesic_radius(d, n_points, k_fraction=k_fraction, percentile=percentile)
    labels = riemannian_clustering_auto(d, radius)
    reps = cluster_means(matrices, labels)
    x3 = embed_3d(reps, random_state)
    metrics = evaluate_embedding_quality_full(reps, x3, knn_k=eval_knn)
    return x3, reps, labels, radius, metrics

# -------------------------
# 5. 3D plotting helper
# -------------------------
def plot_3d_embedding(x3, title, ax):
    ax.scatter(x3[:,0], x3[:,1], x3[:,2], s=50)
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

def plot_3d_embedding_with_cluster_sizes(x3, labels, title, ax, min_size=30, max_size=300, legend=True):
    """
    Scatter plot in 3D where marker size is proportional to cluster size.

    Parameters:
    - x3: (n_clusters, 3) embedding
    - labels: cluster labels for all original points
    - ax: matplotlib 3D axis
    - min_size, max_size: range of marker sizes
    """
    # Count points in each cluster
    unique_labels = np.unique(labels)
    counts = np.array([np.sum(labels == lab) for lab in unique_labels])

    # Scale counts to marker sizes
    sizes = min_size + (counts - counts.min()) / (counts.max() - counts.min() + 1e-9) * (max_size - min_size)

    # Plot each cluster representative
    for i, lab in enumerate(unique_labels):
        ax.scatter(x3[i,0], x3[i,1], x3[i,2], s=sizes[i], label=f"Cluster {lab} ({counts[i]})")

    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    if legend: ax.legend(fontsize=8)

# -------------------------
# 6. Example usage
# -------------------------

band = 0

nband, nrun, ntask, channels, _ = run_centroids.shape
data = run_centroids.reshape(nband, nrun * ntask, channels, channels)

matrices = data[band]

x3, reps, labels, radius, metrics = process_band_task_auto_adaptive(
    matrices,
    k_fraction=0.07,
    percentile=60,
    random_state=42,
    eval_knn=10
)

print("Number of clusters:", len(reps))
print("Estimated radius:", radius)
print("Distance distortion:", metrics["distance_distortion"])
print("kNN preservation:", metrics["knn_preservation"])
print("Angle distortion:", metrics["angle_distortion"])
print("Variance preservation:", metrics["variance_preservation"])

In [ ]:
plt.close()

# 3D plot
plt.close()
fig = plt.figure(figsize=(12, 12))
ax = fig.add_subplot(111, projection='3d')
plot_3d_embedding_with_cluster_sizes(
    x3, 
    labels, 
    f"Band {band} | Clusters={len(reps)}", 
    ax,
    min_size=1, 
    max_size=300,
    legend=False
)
plt.show()



# CLUSTERS

In [ ]:
from scipy.io import savemat, loadmat
import riemann_utils.matrix_functions as mrtrf

import matplotlib
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import hdbscan


loadDay = False
saveFigure = False


# pathDataCentr = '/home/palatella/workspace/cybathlon_user/'
pathDataCentr = pathData


# fileName2019 = 'run_centroids_user_1band.mat' 
# fileName2023 = 'run_centroids_user_20232024_20192020ref_1band.mat'
# keyName = 'run_centroids'

# if loadDay:
#     fileName2019 = fileName2019.replace('run_', 'day_')
#     fileName2023 = fileName2023.replace('run_', 'day_')
#     keyName = 'day_centroids'

# centroids2019 = loadmat(f'{pathDataCentr}/{fileName2019}')
# centroids2023 = loadmat(f'{pathDataCentr}/{fileName2023}')


# mAbsDev_centroids2019 = centroids2019['mAbsDev_centroids']
# centroids2019 = centroids2019[keyName]


# mAbsDev_centroids2023 = centroids2023['mAbsDev_centroids']
# centroids2023 = centroids2023[keyName]

# centroids = np.concatenate((centroids2019, centroids2023), axis=1)
# mAbsDev_centroids = np.concatenate((mAbsDev_centroids2019, mAbsDev_centroids2023), axis=1)


# matrix_distance = mrtrf.compute_runMatrix_distances(centroids, mAbsDev_centroids)
# matrix_angle = mrtrf.compute_runMatrix_angles(centroids, np.eye(centroids.shape[-1]))



fileNameDistance = 'matrix_run_distance_user.mat' if not loadDay else 'matrix_day_distance_user.mat'
fileNameAngle = 'matrix_run_angle_user.mat' if not loadDay else 'matrix_day_angle_user.mat'

fileNameDistance = fileNameDistance.replace('.mat', '_1band.mat')
fileNameAngle = fileNameAngle.replace('.mat', '_1band.mat')

# savemat(f'{pathData}/{fileNameDistance}', {'matrix_distance': matrix_distance})
# savemat(f'{pathData}/{fileNameAngle}', {'matrix_angle': matrix_angle})

matrix_distance = loadmat(f"{pathData}/{fileNameDistance}")['matrix_distance']
matrix_angle_rad = loadmat(f"{pathData}/{fileNameAngle}")['matrix_angle']

matrix_distance[np.isnan(matrix_distance)] = 0
matrix_distance = np.mean(matrix_distance, axis=3)

matrix_angle_rad[np.isnan(matrix_angle_rad)] = 0
matrix_angleRad = np.mean(matrix_angle_rad, axis=3)
matrix_angleDeg = np.rad2deg(matrix_angleRad)
matrix_angleCos = np.cos(matrix_angleRad)
# matrix_mask = np.full(matrix_angle.shape, False)

cluster_labels = np.empty((matrix_distance.shape[0], matrix_distance.shape[1]), dtype=int)
idx_cluster = np.empty(cluster_labels.shape)
idx_cluster_split = np.empty(matrix_distance.shape[0], dtype=object)

In [8]:
def normalize_matrix(M):
    M = M.copy()
    nonzero = M[M > 0]
    scale = np.median(nonzero)
    return M / scale

def build_affinity(D, A, beta=1.0):
    W = np.exp(-D) * np.exp(-beta * A)
    np.fill_diagonal(W, 0.0)
    return W

def diffusion_maps(W, n_eigs=15, alpha=0.5):
    """
    alpha = 0.5 removes sampling density bias
    """
    d = W.sum(axis=1)
    D_alpha = np.diag(d**(-alpha))
    K = D_alpha @ W @ D_alpha

    d_tilde = K.sum(axis=1)
    P = np.diag(1.0 / d_tilde) @ K  # diffusion operator

    eigvals, eigvecs = np.linalg.eigh(P)
    idx = np.argsort(eigvals)[::-1]

    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    return eigvals[:n_eigs], eigvecs[:, :n_eigs]


def diffusion_embedding(eigvals, eigvecs, t=1, m=3):
    lambdas = eigvals[1:m+1]
    psi = eigvecs[:, 1:m+1]
    return psi * (lambdas ** t)

In [9]:
from sklearn.preprocessing import StandardScaler
x_embeddings = np.empty((matrix_distance.shape[0], matrix_distance.shape[1], 3))

for band in range(matrix_distance.shape[0]):
    cluster_data_angleRad = matrix_angleRad[band]
    cluster_data_distance = matrix_distance[band]

    np.fill_diagonal(cluster_data_angleRad, val=0)
    np.fill_diagonal(cluster_data_distance, val=0)

    D = normalize_matrix(cluster_data_distance)
    A = normalize_matrix(cluster_data_angleRad)

    W = build_affinity(D, A, beta=1.0)

    eigvals, eigvecs = diffusion_maps(W)

    print(f"Leading diffusion eigenvalues for band {band+1}:")
    for i, v in enumerate(eigvals[:10]):
        print(f"{i}: {v:.4f}")

    X_diff = diffusion_embedding(eigvals, eigvecs, t=1, m=3)
    x_embeddings[band] = X_diff


    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=int(0.05 * X_diff.shape[0]),
        min_samples=5,
        metric='euclidean'
    )

    cl_labels = clusterer.fit_predict(X_diff)


    X_diff_t2 = diffusion_embedding(eigvals, eigvecs, t=2, m=3)

    labels_t2 = hdbscan.HDBSCAN(
        min_cluster_size=int(0.05 * X_diff.shape[0]),
        min_samples=5
    ).fit_predict(X_diff_t2)


    stable = (cl_labels == labels_t2) & (cl_labels != -1)
    final_labels = cl_labels.copy()
    final_labels[~stable] = -1
    # cluster_labels[band] = final_labels
    cluster_labels[band] = labels_t2



    t_idx = np.empty(0)
    split_idx = []
    for lb in np.unique(cluster_labels[band]):
        t_idx = np.append(t_idx, np.where(cluster_labels[band] == lb)[0])  
        split_idx.append(t_idx.shape[0])
    idx_cluster[band] = t_idx
    idx_cluster_split[band] = split_idx

Leading diffusion eigenvalues for band 1:
0: 0.9956
1: 0.5150
2: 0.3224
3: 0.1587
4: 0.1381
5: 0.1121
6: 0.0656
7: 0.0550
8: 0.0517
9: 0.0420


# ANGLE MATRICES

In [ ]:
day_start = (
    user
    .drop_duplicates(subset='nDay', keep='first')
    .sort_index()
)

day_start_idx = day_start['nRun'].to_numpy()
day_labels = day_start['dayLabel'].to_numpy()

if loadDay: day_start_idx = np.array(range(len(day_labels)))

labels = []
for i, lab in enumerate(day_labels):
    if i % 3== 0:
        labels.append(lab)
    else:
        labels.append('')

days_modelCreation = [
    "02/05/2019",
    "09/07/2019",
    "27/10/2020",
    "03/10/2023",
    "11/10/2024",
    "14/10/2024",
    "03/10/2023",
    "02/11/2023",
    "28/11/2023",
    "07/12/2023",
    "19/12/2023",
    "08/07/2024",
    "02/10/2024",
    "04/10/2024",
    "07/10/2024"
]
stop_training = ['10/09/2020','14/10/2024', '05/10/2023']  
stop_idx = np.array([next((day_start_idx[i] for i, s in enumerate(day_labels) if s == pattern), np.nan) for pattern in stop_training])
recalibrations = days_modelCreation[1:]
rec_idx = np.array([next((day_start_idx[i] for i, s in enumerate(day_labels) if s == pattern), np.nan) for pattern in recalibrations])

In [ ]:
accuracy = user.groupby('nRun')['accuracy'].mean().to_numpy()
rejection = user.groupby('nRun')['rejection'].mean().to_numpy()

In [14]:
saveFigure = False
doCluster = False

In [20]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import matplotlib

def get_x(scatter_data):
    return np.arange(len(scatter_data))


def plot_similarity_matrices(
    matrix_angleCos,
    matrix_distance,
    cluster_labels,
    n_bands,
    n_run,
    day_start_idx,
    labels,
    stop_idx=None,
    rec_idx=None,
    doCluster=False,
    idx_cluster=None,
    idx_cluster_split=None,
    cmap_name='PuBu',
    saveFigure=False,
    savingPath='.',
    filename='aaaa.svg',
    figsize=(18, 9)
):
    """
    Plot similarity matrices with optional clustering and separate colorbars.

    Row 0: scatter (no colorbar, reserved space)
    Row 1: angle cosine matrices + colorbar
    Row 2: distance matrices + colorbar
    """

    stop_idx = stop_idx if stop_idx is not None else []
    rec_idx = rec_idx if rec_idx is not None else []

    plt.close('all')

    cmap = matplotlib.colormaps[cmap_name]

    # ---------------- FIGURE & GRIDSPEC ----------------
    fig = plt.figure(figsize=figsize)
    # fig.set_constrained_layout(True)


    gs = gridspec.GridSpec(
        3, n_bands + 1,
        width_ratios=[1] * n_bands + [0.05],
        height_ratios=[1, 5, 5],
        hspace=0.05,
        wspace=0.1
    )

    axes_top1 = []
    # axes_top2 = []
    axes_angle = []
    axes_dist = []

    for i in range(n_bands):
        ax_t1 = fig.add_subplot(gs[0, i])
        # ax_t2 = fig.add_subplot(gs[1, i], sharex=ax_t1)
        # ax_a  = fig.add_subplot(gs[2, i], sharex=ax_t1)
        # ax_d  = fig.add_subplot(gs[3, i], sharex=ax_t1)
        ax_a  = fig.add_subplot(gs[1, i], sharex=ax_t1)
        ax_d  = fig.add_subplot(gs[2, i], sharex=ax_t1)

        axes_top1.append(ax_t1)
        # axes_top2.append(ax_t2)
        axes_angle.append(ax_a)
        axes_dist.append(ax_d)


    # ---------------- COLORBAR AXES ----------------
    cax_top1  = fig.add_subplot(gs[0, -1])
    # cax_top2  = fig.add_subplot(gs[1, -1])
    # cax_angle = fig.add_subplot(gs[2, -1])
    # cax_dist  = fig.add_subplot(gs[3, -1])
    cax_angle = fig.add_subplot(gs[1, -1])
    cax_dist  = fig.add_subplot(gs[2, -1])

    cax_top1.axis('off')
    # cax_top2.axis('off')


    # ---------------- CLUSTERED DATA ----------------
    angle_data = matrix_angleCos.copy()
    dist_data = matrix_distance.copy()

    if doCluster:
        angle_tmp = np.empty_like(angle_data)
        dist_tmp = np.empty_like(dist_data)

        for i in range(n_bands):
            idx = idx_cluster[i].astype(int)
            angle_tmp[i] = angle_data[i][idx][:, idx]
            dist_tmp[i] = dist_data[i][idx][:, idx]

        angle_data = angle_tmp
        dist_data = dist_tmp

    # ---------------- TOP ROW: SCATTER ----------------
    for i, ax in enumerate(axes_top1):
        scatter_data = cluster_labels[i]

        if doCluster:
            scatter_data = scatter_data[idx_cluster[i].astype(int)]

        x_scatter = get_x(scatter_data)

        ax.scatter(x_scatter, scatter_data, s=4, color='g')
        ax.set_xlim(-0.5, len(x_scatter) - 0.5)

        ax.set_ylim(-1.5, scatter_data.max() + 0.5)
        ax.set_xlim(-0.5, n_run - 0.5)
        ax.tick_params(axis='x', bottom=False, labelbottom=False)
        ax.tick_params(axis='y', bottom=False, labelbottom=False)


        if not doCluster:
            ax.set_title(f'Band {i}')
            for idx in day_start_idx:
                ax.axvline(idx - 0.5, color='k', lw=0.5, alpha=0.3)
            for idx in stop_idx:
                ax.axvline(idx - 0.5, color='r', lw=1.0)
            for idx in rec_idx:
                ax.axvline(idx - 0.5, color='orange', lw=1.0)
        else:
            ax.set_yticks([])
            for idx in idx_cluster_split[i]:
                ax.axvline(idx - 0.5, color='r', lw=1.0)

    # for i, ax in enumerate(axes_top2):
    #     scatter_accuracy = accuracy
    #     scatter_rejection = rejection

    #     if doCluster:
    #         scatter_accuracy = scatter_accuracy[idx_cluster[i].astype(int)]
    #         scatter_rejection = scatter_rejection[idx_cluster[i].astype(int)]

    #     x_scatter = get_x(scatter_accuracy)

    #     ax.scatter(x_scatter, scatter_accuracy, s=4, color='g')
    #     ax.scatter(x_scatter, scatter_rejection, s=4, color='r')
    #     ax.set_xlim(-0.5, len(x_scatter) - 0.5)


    #     ax.set_ylim(0, 1)
    #     ax.tick_params(axis='x', bottom=False, labelbottom=False)

    #     if not doCluster:
    #         for idx in day_start_idx:
    #             ax.axvline(idx - 0.5, color='k', lw=0.5, alpha=0.3)
    #         for idx in stop_idx:
    #             ax.axvline(idx - 0.5, color='r', lw=1.0)
    #         for idx in rec_idx:
    #             ax.axvline(idx - 0.5, color='orange', lw=1.0)
    #     else:
    #         ax.set_yticks([])
    #         for idx in idx_cluster_split[i]:
    #             ax.axvline(idx - 0.5, color='r', lw=1.0)


    # ---------------- ANGLE MATRICES ----------------
    for i, ax in enumerate(axes_angle):
        im_angle = ax.imshow(
            angle_data[i],
            cmap=cmap.reversed(),
            vmin=0,
            vmax=1,
            aspect='auto'
        )
        # ax.set_aspect('equal', adjustable='box')
        ax.tick_params(axis='x', which='both', bottom=True, labelbottom=False)


        if not doCluster:
            ax.set_xticks(day_start_idx - 0.5)
            ax.set_yticks(day_start_idx - 0.5)

            # ax.set_xticklabels(labels, rotation=45, ha='right')
            ax.set_yticklabels(labels if i == 0 else [])

            for idx in day_start_idx:
                ax.axvline(idx - 0.5, color='k', lw=0.5, alpha=0.3)
                ax.axhline(idx - 0.5, color='k', lw=0.5, alpha=0.3)

            for idx in stop_idx:
                ax.axvline(idx - 0.5, color='r', lw=1.0)
                ax.axhline(idx - 0.5, color='r', lw=1.0)

            for idx in rec_idx:
                ax.axvline(idx - 0.5, color='orange', lw=1.0)
                ax.axhline(idx - 0.5, color='orange', lw=1.0)
        else:
            ax.set_xticks([])
            ax.set_yticks([])
            for idx in idx_cluster_split[i]:
                ax.axvline(idx - 0.5, color='r', lw=1.0)
                ax.axhline(idx - 0.5, color='r', lw=1.0)

    fig.colorbar(im_angle, cax=cax_angle, label='Angle Cosine')

    # ---------------- DISTANCE MATRICES ----------------
    for i, ax in enumerate(axes_dist):
        im_dist = ax.imshow(
            dist_data[i],
            cmap=cmap,
            vmin=0,
            vmax=2,
            aspect='auto'
        )
        # ax.set_aspect('equal', adjustable='box')

        if not doCluster:
            ax.set_xticks(day_start_idx - 0.5)
            ax.set_yticks(day_start_idx - 0.5)

            ax.set_xticklabels(labels, rotation=45, ha='right')
            ax.set_yticklabels(labels if i == 0 else [])

            for idx in day_start_idx:
                ax.axvline(idx - 0.5, color='k', lw=0.5, alpha=0.3)
                ax.axhline(idx - 0.5, color='k', lw=0.5, alpha=0.3)

            for idx in stop_idx:
                ax.axvline(idx - 0.5, color='r', lw=1.0)
                ax.axhline(idx - 0.5, color='r', lw=1.0)

            for idx in rec_idx:
                ax.axvline(idx - 0.5, color='orange', lw=1.0)
                ax.axhline(idx - 0.5, color='orange', lw=1.0)
        else:
            ax.set_xticks([])
            ax.set_yticks([])
            for idx in idx_cluster_split[i]:
                ax.axvline(idx - 0.5, color='r', lw=1.0)
                ax.axhline(idx - 0.5, color='r', lw=1.0)

    fig.colorbar(im_dist, cax=cax_dist, label='Geodesic Distance')

    # for ax in axes_top1 + axes_top2:
    #     ax.set_aspect('auto')

    # for ax in axes_angle + axes_dist:
    #     ax.set_aspect('equal', adjustable='box')

    # fig.set_constrained_layout(True)
    plt.show()

    if saveFigure:
        fullpath = f"{savingPath}/{filename}"
        print(f"Saving figure to {fullpath}")
        fig.savefig(fullpath, bbox_inches='tight', dpi=800, format='svg')


In [ ]:
top_height = 1.2   # per scatter row
cell_size = 10
cbar_width = 0.4

t_cluster_lbl = cluster_labels.copy()
# t_cluster_lbl[1] = t_cluster_lbl[0]

t_idx_cluster = idx_cluster.copy()
# t_idx_cluster[1] = t_idx_cluster[0]

t_idx_cluster_split = idx_cluster_split.copy()
# t_idx_cluster_split[1] = t_idx_cluster_split[0]

n_bands = matrix_angleCos.shape[0]
fig_width = n_bands * cell_size + cbar_width
fig_height = 2 * top_height + 2 * cell_size

doCluster = False
saveFigure = True

plot_similarity_matrices(
    matrix_angleCos=matrix_angleCos,
    matrix_distance=matrix_distance,
    cluster_labels=t_cluster_lbl,
    n_bands=n_bands,
    n_run=matrix_angleCos.shape[1],
    day_start_idx=day_start_idx,
    labels=labels,
    stop_idx=stop_idx,
    rec_idx=rec_idx,
    doCluster=doCluster,
    idx_cluster=t_idx_cluster,
    idx_cluster_split=t_idx_cluster_split,
    saveFigure=saveFigure,
    savingPath=savingPath,
    filename='similarityNoGroupun.svg',
    figsize=(fig_width, fig_height)
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
from matplotlib.patches import Patch


# -------------------- Helpers --------------------
def make_cluster_colormap(labels, base_cmap='tab20'):
    unique = np.unique(labels[labels != -1])
    cmap = plt.get_cmap(base_cmap, len(unique))
    cluster_color = {cl: cmap(i) for i, cl in enumerate(unique)}
    cluster_color[-1] = (0.85, 0.85, 0.85, 1.0)  # noise
    return cluster_color


def cluster_bar_x(labels, cluster_color):
    bar = np.zeros((1, len(labels), 4))
    for i, lb in enumerate(labels):
        bar[0, i] = cluster_color[lb]
    return bar


def cluster_bar_y(labels, cluster_color):
    bar = np.zeros((len(labels), 1, 4))
    for i, lb in enumerate(labels):
        bar[i, 0] = cluster_color[lb]
    return bar

from matplotlib.patches import Rectangle

def fixed_cluster_colors(labels):
    base_colors = [
        (0.84, 0.15, 0.16, 1.0),  # red
        (0.12, 0.47, 0.71, 1.0),  # blue
        (0.17, 0.63, 0.17, 1.0),  # green
        (0.58, 0.40, 0.74, 1.0),  # purple
        (1.00, 0.50, 0.05, 1.0),  # orange
    ]

    cluster_color = {-1: (0.7, 0.7, 0.7, 1.0)}  # gray for noise

    for lb in np.unique(labels):
        if lb == -1:
            continue
        cluster_color[lb] = base_colors[lb % len(base_colors)]

    return cluster_color


def cluster_bar_vertical(labels, cluster_color):
    bar = np.zeros((len(labels), 1, 4))
    for i, lb in enumerate(labels):
        bar[i, 0] = cluster_color[lb]
    return bar


def cluster_bar_horizontal(labels, cluster_color):
    bar = np.zeros((1, len(labels), 4))
    for i, lb in enumerate(labels):
        bar[0, i] = cluster_color[lb]
    return bar


# def draw_bar_outline(ax, nrows, ncols):
#     rect = Rectangle(
#         (-0.5, -0.5),
#         ncols,
#         nrows,
#         fill=False,
#         lw=1.2,
#         edgecolor='black'
#     )
#     ax.add_patch(rect)

# def draw_bar_outline_from_image(ax, img):
#     nrows, ncols = img.shape[:2]

#     # Force axis limits to match image pixels
#     ax.set_xlim(-0.5, ncols - 0.5)
#     ax.set_ylim(nrows - 0.5, -0.5)

#     rect = Rectangle(
#         (-0.5, -0.5),
#         ncols,
#         nrows,
#         fill=False,
#         lw=1.5,
#         edgecolor='black'
#     )
#     ax.add_patch(rect)

# def draw_bar_outline_exact(ax, nrows, ncols):
#     rect = Rectangle(
#         (0, 0),
#         ncols,
#         nrows,
#         fill=False,
#         lw=1.5,
#         edgecolor='black'
#     )
#     ax.add_patch(rect)

def plot_similarity_matrices_andClusters(
    matrix_angleCos,
    matrix_distance,
    cluster_labels,
    n_bands,
    n_run,
    day_start_idx,
    labels,
    stop_idx=None,
    rec_idx=None,
    doCluster=False,
    idx_cluster=None,
    idx_cluster_split=None,
    cmap_name='PuBu',
    saveFigure=False,
    savingPath='.',
    filename='similarity_marginal_clusters.svg',
    figsize=(18, 10)
):

    stop_idx = stop_idx if stop_idx is not None else []
    rec_idx  = rec_idx if rec_idx is not None else []

    plt.close('all')
    cmap = matplotlib.colormaps[cmap_name]

    fig = plt.figure(figsize=figsize)

    gs = gridspec.GridSpec(
        2, n_bands + 1,
        width_ratios=[1]*n_bands + [0.04],
        height_ratios=[1, 1],
        wspace=0.15,
        hspace=0.15
    )

    cax_angle = fig.add_subplot(gs[0, -1])
    cax_dist  = fig.add_subplot(gs[1, -1])

    im_angle = None
    im_dist  = None

    for b in range(n_bands):

        lbl = cluster_labels[b]
        if doCluster:
            lbl = lbl[idx_cluster[b].astype(int)]

        cluster_color = fixed_cluster_colors(lbl)

        for row, (data, vmin, vmax) in enumerate([
            (matrix_angleCos[b], 0, 1),
            (matrix_distance[b],  0, 2)
        ]):

            subgs = gridspec.GridSpecFromSubplotSpec(
                2, 2,
                subplot_spec=gs[row, b],
                width_ratios=[1, 0.06],
                height_ratios=[0.06, 1],
                wspace=0.0,
                hspace=0.0
            )

            ax_xbar = fig.add_subplot(subgs[0, 0])
            ax_ybar = fig.add_subplot(subgs[1, 1])
            ax_mat  = fig.add_subplot(subgs[1, 0])

            # ---------------- MATRIX ----------------
            # im = ax_mat.imshow(
            #     data,
            #     cmap=cmap.reversed() if row == 0 else cmap,
            #     vmin=vmin,
            #     vmax=vmax,
            #     interpolation='nearest',
            #     aspect='auto'
            # )

            # n = data.shape[0]
            # ax_mat.set_xlim(0, n)
            # ax_mat.set_ylim(n, 0)

            n = data.shape[0]

            im = ax_mat.imshow(
                data,
                cmap=cmap.reversed() if row == 0 else cmap,
                vmin=vmin,
                vmax=vmax,
                interpolation='nearest',
                origin='upper',
                extent=[0, n, n, 0],   # FORCE PIXEL GRID
                aspect='auto'
            )

            ax_mat.add_patch(Rectangle(
                (0, 0), 1, 1,
                transform=ax_mat.transAxes,
                fill=False,
                lw=1.2,
                edgecolor='black',
                antialiased=False,
                snap=True,
                zorder=1000
            ))

            

            ax_mat.set_xlim(0, n)
            ax_mat.set_ylim(n, 0)


            if row == 0 and im_angle is None:
                im_angle = im
            if row == 1 and im_dist is None:
                im_dist = im

            # ---------------- TOP BAR ----------------
            bar_h = cluster_bar_horizontal(lbl, cluster_color)
            _, ncols = bar_h.shape[:2]

            ax_xbar.imshow(
                bar_h,
                origin='upper',
                extent=[0, ncols, 1, 0],
                interpolation='nearest',
                aspect='auto'
            )
            ax_xbar.set_xlim(ax_mat.get_xlim())
            ax_xbar.set_ylim(1, 0)

            # ax_xbar.add_patch(Rectangle(
            #     (0, 0), ncols, 1,
            #     fill=False,
            #     lw=1.2,
            #     edgecolor='black',
            #     antialiased=False,
            #     snap=True
            # ))
            ax_xbar.add_patch(Rectangle(
                (0, 0), 1, 1,
                transform=ax_xbar.transAxes,   # AXES COORDS
                fill=False,
                lw=1.2,
                edgecolor='black',
                antialiased=False,
                snap=True,
                zorder=1000
            ))



            ax_xbar.set_xticks([])
            ax_xbar.set_yticks([])

            # ---------------- RIGHT BAR ----------------
            bar_v = cluster_bar_vertical(lbl, cluster_color)
            nrows, _ = bar_v.shape[:2]

            ax_ybar.imshow(
                bar_v,
                origin='upper',
                extent=[0, 1, nrows, 0],
                interpolation='nearest',
                aspect='auto'
            )
            ax_ybar.set_xlim(0, 1)
            ax_ybar.set_ylim(ax_mat.get_ylim())


            # ax_ybar.add_patch(Rectangle(
            #     (0, 0), 1, nrows,
            #     fill=False,
            #     lw=1.2,
            #     edgecolor='black',
            #     antialiased=False,
            #     snap=True
            # ))
            ax_ybar.add_patch(Rectangle(
                (0, 0), 1, 1,
                transform=ax_ybar.transAxes,   # AXES COORDS
                fill=False,
                lw=1.2,
                edgecolor='black',
                antialiased=False,
                snap=True,
                zorder=1000
            ))



            ax_ybar.set_xticks([])
            ax_ybar.set_yticks([])

            # ---------------- LABELS ----------------
            # if row == 0:
            #     ax_mat.set_title(f'Band {b}', fontsize=11)

            ax_mat.set_xticks(day_start_idx)
            ax_mat.set_yticks(day_start_idx)

            if row == 1:
                ax_mat.set_xticklabels(labels, rotation=45, ha='right')
            else:
                ax_mat.set_xticklabels([])

            ax_mat.set_yticklabels(labels if b == 0 else [])

            # ---------------- TEMPORAL MARKERS ----------------
            if not doCluster:
                for idx in day_start_idx:
                    ax_mat.axvline(idx, color='k', lw=0.5, alpha=0.3,zorder=10)
                    ax_mat.axhline(idx, color='k', lw=0.5, alpha=0.3,zorder=10)

                for idx in stop_idx:
                    ax_mat.axvline(idx, color='r', lw=1.0,zorder=10)
                    ax_mat.axhline(idx, color='r', lw=1.0,zorder=10)

                for idx in rec_idx:
                    ax_mat.axvline(idx, color='orange', lw=1.0,zorder=10)
                    ax_mat.axhline(idx, color='orange', lw=1.0,zorder=10)

            # for ax in (ax_xbar, ax_ybar, ax_mat):
            #     for spine in ax.spines.values():
            #         spine.set_visible(False)


    # ---------------- COLORBARS ----------------
    fig.colorbar(im_angle, cax=cax_angle, label='Angle Cosine')
    fig.colorbar(im_dist,  cax=cax_dist,  label='Geodesic Distance')

    # ---------------- LEGEND ----------------
    legend_items = [
        Patch(facecolor=v, label=f'Cluster {k}')
        for k, v in cluster_color.items() if k != -1
    ]
    legend_items.append(Patch(facecolor=cluster_color[-1], label='Noise'))

    fig.legend(handles=legend_items, loc='upper right', frameon=False)

    plt.show()

    if saveFigure:
        path = f"{savingPath}/{filename}"
        fig.savefig(path, bbox_inches='tight', dpi=800, format='svg')
        print(f"Saved figure to {path}")


In [24]:
np.unique(cluster_labels)

array([-1,  0,  1,  2,  3])

In [ ]:
top_height = 1.2   # per scatter row
cell_size = 10
cbar_width = 0.4

t_cluster_lbl = cluster_labels.copy()
# t_cluster_lbl[1] = t_cluster_lbl[0]

t_idx_cluster = idx_cluster.copy()
# t_idx_cluster[1] = t_idx_cluster[0]

t_idx_cluster_split = idx_cluster_split.copy()
# t_idx_cluster_split[1] = t_idx_cluster_split[0]

n_bands = matrix_angleCos.shape[0]
fig_width = n_bands * cell_size + cbar_width
fig_height = 2 * top_height + 2 * cell_size

doCluster = False
saveFigure = True

plot_similarity_matrices_andClusters(
    matrix_angleCos=matrix_angleCos,
    matrix_distance=matrix_distance,
    cluster_labels=t_cluster_lbl,
    n_bands=n_bands,
    n_run=matrix_angleCos.shape[1],
    day_start_idx=day_start_idx,
    labels=labels,
    stop_idx=stop_idx,
    rec_idx=rec_idx,
    doCluster=doCluster,
    idx_cluster=t_idx_cluster,
    idx_cluster_split=t_idx_cluster_split,
    saveFigure=saveFigure,
    savingPath=savingPath,
    filename='similarity_matrices_grouped.svg',
    figsize=(fig_width, fig_height)
)




In [ ]:
data_angle = matrix_angleRad[0]
data_distance = matrix_distance[0]

def compute_directional_coherence_index(matrix_rad, diag=1):
    return np.mean(np.cos(np.diag(matrix_rad, diag)))

def compute_geodesic_efficency(matrix_dist, diag=1):
    return matrix_dist[0, -1] / np.sum(np.diag(matrix_dist, diag))

##### Directional Coherence Index (DCI) (key metric)
# Interpretation:
# ≈ 1 → strong consistent direction
# ≈ 0 → random walk
# < 0 → oscillatory / back-and-forth
# This is precisely what “direction” means on a manifold.
DCI = np.mean(np.cos(np.diag(data_angle, 1)))
print("Directional Coherence Index:", DCI)

## Geodesic Efficiency (Drift Ratio)
# Measures whether motion is straight or meandering.
# ≈ 1 → straight geodesic-like evolution
# ≪ 1 → reversals / wandering
GE = data_distance[0, -1] / np.sum(np.diag(data_distance, 1))
print("Geodesic Efficiency:", GE)


Directional Coherence Index: 0.7847473585910056
Geodesic Efficiency: 0.006874367380875981


In [44]:
def compute_distance_directional_coherence(matrix_dist, diag=1):
    return np.mean(np.diag(matrix_dist, diag))

def compute_angle_directional_coherence(matrix_angleRad, diag=1):
    return np.mean(np.diag(np.cos(matrix_angleRad), diag))

In [57]:
adc = np.empty(cluster_labels.shape[0], dtype=object)
ddc = np.empty(cluster_labels.shape[0], dtype=object)

for band in range(cluster_labels.shape[0]):
    unique = np.unique(cluster_labels[band])
    t_dci = np.empty((len(unique)+1), dtype=object)
    t_ge = np.empty((len(unique)+1), dtype=object)
    for idx_lab, lab in enumerate(unique):
        idx = np.where(cluster_labels[band] == lab)[0]
        print(f"Band {band} | Cluster {lab} | Size {len(idx)}")
        if len(idx) > 1:
            sub_angle = matrix_angleRad[band][np.ix_(idx, idx)]
            sub_dist = matrix_distance[band][np.ix_(idx, idx)]
            t_t_dci = []
            t_t_ge = []
            for k in range(1,len(idx)-1):
                t_t_dci.append(compute_angle_directional_coherence(sub_angle, diag=k))
                t_t_ge.append(compute_distance_directional_coherence(sub_dist, diag=k))
            t_dci[idx_lab] = np.array(t_t_dci)
            t_ge[idx_lab] = np.array(t_t_ge)

    t_t_dci = []
    t_t_ge = []
    for k in range(1,matrix_angleRad.shape[-1]):
        t_t_dci.append(compute_angle_directional_coherence(matrix_angleRad[band], diag=k))
        t_t_ge.append(compute_distance_directional_coherence(matrix_distance[band], diag=k))
    t_dci[-1] = np.array(t_t_dci)
    t_ge[-1] = np.array(t_t_ge)

    adc[band] = np.array(t_dci)
    ddc[band] = np.array(t_ge)

Band 0 | Cluster -1 | Size 18
Band 0 | Cluster 0 | Size 40
Band 0 | Cluster 1 | Size 18
Band 0 | Cluster 2 | Size 171
Band 0 | Cluster 3 | Size 13


In [ ]:
# def plot_dci_per_band(
#     band_data,
#     ylim=(-0.1, 1.5),
#     max_cols=4,
#     cell_width=4,
#     cell_height=3,
#     markersize=4,
#     alpha=0.8,
#     legend_ncol=8
# ):
#     """
#     Plot DCI per band: one subplot per band, each cluster as a separate line.
#     Returns (fig, axes).
#     """
#     n_bands = len(band_data)
#     if n_bands == 0:
#         return None, None

#     ncols = min(max_cols, n_bands)
#     nrows = int(np.ceil(n_bands / ncols))
#     figsize = (ncols * cell_width, nrows * cell_height)

#     plt.close('all')
#     fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
#     axes = np.atleast_1d(axes).reshape(-1)

#     handles, labels = [], []
#     for b in range(n_bands):
#         ax = axes[b]
#         clusters = band_data[b]
#         if clusters is None or len(clusters) == 0:
#             ax.axis('off')
#             continue
#         for i, c in enumerate(clusters):
#             arr = np.asarray(c)
#             if arr.size == 0:
#                 continue
#             h, = ax.plot(np.arange(len(arr)), arr, marker='o', markersize=markersize,
#                          label=f'cls {i}', alpha=alpha)
#             handles.append(h)
#             labels.append(f'cls {i}')
#         ax.set_title(f'Band {b}')
#         ax.set_xlabel('lag')
#         ax.grid(alpha=0.3)
#         if ylim is not None:
#             ax.set_ylim(ylim)

#     # hide unused axes
#     for ax in axes[n_bands:]:
#         ax.axis('off')

#     # shared y-label and consolidated legend (deduplicated preserving order)
#     fig.text(0.04, 0.5, 'DCI', va='center', rotation='vertical')

#     seen = set()
#     uniq_handles, uniq_labels = [], []
#     for h, l in zip(handles, labels):
#         if l not in seen:
#             seen.add(l)
#             uniq_handles.append(h)
#             uniq_labels.append(l)

#     if uniq_handles:
#         fig.legend(uniq_handles, uniq_labels, loc='upper center',
#                    ncol=min(legend_ncol, len(uniq_labels)), fontsize='small',
#                    bbox_to_anchor=(0.5, 1.02))

#     plt.tight_layout(rect=[0, 0, 1, 0.96])
#     plt.show()
#     return fig, axes

# plot_dci_per_band(
#     adc,
#     ylim=(-0.1, 1.5),
#     max_cols=4,
#     cell_width=10,
#     cell_height=8,
#     markersize=4,
#     alpha=0.8,
#     legend_ncol=8)

In [ ]:
def plot_dci_single_band(
    clusters,
    clter_colors,
    ylim=(-0.1, 1.5),
    cell_width=10,
    cell_height=8,
    markersize=4,
    alpha=0.8,
    legend_ncol=8,
):
    """
    Plot DCI for a single band: one plot, each cluster as a separate line.
    Returns (fig, ax).
    """
    if clusters is None or len(clusters) == 0:
        return None, None

    plt.close('all')
    fig, ax = plt.subplots(figsize=(cell_width, cell_height))

    for i, c in enumerate(clusters):
        arr = np.asarray(c)
        if arr.size == 0:
            continue

        lbl = f'cls {i-1}' if i > 0 else 'noise' 
        if i == len(clusters) - 1:
            lbl = 'all data'
    
        ax.plot(
            np.arange(len(arr)),
            arr,
            marker='o',
            markersize=markersize,
            label=lbl,
            alpha=alpha,
            color=clter_colors[i-1]
        )

    ax.set_title('Distance Directional Coherence per Cluster')
    ax.set_xlabel('lag')
    ax.set_ylabel('DDC')
    ax.grid(alpha=0.3)

    if ylim is not None:
        ax.set_ylim(ylim)

    ax.set_xlim(0, 259)
    


    ax.legend(
        loc='upper center',
        ncol=min(legend_ncol, len(clusters)),
        fontsize='small'
    )

    plt.tight_layout()
    plt.show()

    return fig, ax

clst_color = fixed_cluster_colors(list(range(len(adc[0]))))
fig, ax = plot_dci_single_band(
    ddc[0],          # now directly clusters for the single band
    clter_colors=clst_color,
    ylim=(0, 1.5),
    cell_width=13,
    cell_height=6,
    markersize=4,
    alpha=0.8,
    legend_ncol=8
)

path = f"{savingPath}/ddc_per_cluster.svg"
fig.savefig(path, bbox_inches='tight', dpi=800, format='svg')
print(f"Saved figure to {path}")


In [26]:


def compute_transition_matrix(labels):
    valid = labels[labels != -1]
    states = np.unique(valid)
    idx = {s: i for i, s in enumerate(states)}

    T = np.zeros((len(states), len(states)))

    for t in range(len(labels) - 1):
        if (labels[t] != -1) and (labels[t+1] != -1):
            i = idx[labels[t]]
            j = idx[labels[t+1]]
            T[i, j] += 1

    T = T / np.maximum(T.sum(axis=1, keepdims=True), 1)
    return T, states

# “Clustering is performed in a geometry-driven, order-invariant manner to identify recurrent SPD states. Temporal structure is subsequently 
# analyzed by studying the evolution of cluster assignments along the original time axis.”

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

saveFigure = True


plt.close('all')
colors_cluster = fixed_cluster_colors(cluster_labels[0])
for band in range(matrix_distance.shape[0]):

    t_lables = cluster_labels[band].copy()
    X_diff = x_embeddings[band]  # shape (n_time, 3)

    time = np.arange(len(t_lables))

    T, states = compute_transition_matrix(t_lables)

    lbl = t_lables

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')

    # ---- plot trajectory (time) ----
    for t in range(len(X_diff) - 1):
        ax.plot(
            X_diff[t:t+2, 0],
            X_diff[t:t+2, 1],
            X_diff[t:t+2, 2],
            color='gray',
            alpha=0.3,
            linewidth=1
        )

    # ---- plot clusters ----
    for lab in np.unique(lbl):
        mask = lbl == lab

        if lab == -1:
            ax.scatter(
                X_diff[mask, 0],
                X_diff[mask, 1],
                X_diff[mask, 2],
                c='lightgray',
                s=20,
                label='noise',
                edgecolors='k'

            )
        else:
            ax.scatter(
                X_diff[mask, 0],
                X_diff[mask, 1],
                X_diff[mask, 2],
                s=35,
                label=f'cluster {lab}',
                edgecolors='k',
                color=colors_cluster[lab]

            )

    ax.set_title("Temporal Trajectory in 3D Diffusion Space (SPD geometry)")
    ax.set_xlabel("Diffusion coord 1")
    ax.set_ylabel("Diffusion coord 2")
    ax.set_zlabel("Diffusion coord 3")
    ax.legend()
    plt.tight_layout()
    plt.show()

    if saveFigure:
        filename = f"diffusionMap3d_band{band}.svg"
        fullpath = f"{savingPath}/{filename}"
        print(f"Saving figure to {fullpath}")
        fig.savefig(fullpath, bbox_inches='tight', dpi=800, format='svg')

    import matplotlib.pyplot as plt

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')

    sc = ax.scatter(
        X_diff[:, 0],
        X_diff[:, 1],
        X_diff[:, 2],
        c=time,
        cmap='viridis',
        s=40,
        edgecolors='k'
    )
    cbar = fig.colorbar(sc, ax=ax, pad=0.1)
    cbar.set_ticks([])
    cbar.ax.yaxis.set_ticklabels([])
    cbar.outline.set_visible(False)
    cbar.set_label("Time index")
    plt.title("Diffusion Space Colored by Time")
    plt.xlabel("Diffusion coord 1")
    plt.ylabel("Diffusion coord 2")
    plt.show()

    if saveFigure:
        filename = f"time_diffusionMap3d_band{band}.svg"
        fullpath = f"{savingPath}/{filename}"
        print(f"Saving figure to {fullpath}")
        fig.savefig(fullpath, bbox_inches='tight', dpi=800, format='svg')



# MATRIX EVOLUTION

In [ ]:
from pyriemann.utils.tangentspace import log_map_riemann, exp_map_riemann
import riemann_utils.matrix_functions as mrtrf 
from sklearn.datasets import make_spd_matrix

def riemannian_point_with_distance_and_angle(center, m0, symMat=None, dist=2.0, alpha_deg=30.0):
    alpha = np.deg2rad(alpha_deg)

    v0 = log_map_riemann(m0, center)
    u0 = v0 / mrtrf.norm_riemann(v0, center)

    if symMat is None:
        symMat = make_spd_matrix(center.shape[0])

    symMat = log_map_riemann(symMat, center)

    proj = mrtrf.metric_riemann(symMat, u0, center)
    u1 = symMat - proj * u0
    u1 /= mrtrf.norm_riemann(u1, center)

    v = dist * (np.cos(alpha) * u0 + np.sin(alpha) * u1)
    return exp_map_riemann(v, center)

from pyriemann.utils.distance import distance_riemann


# Dimension
n = 13

center = make_spd_matrix(n)
m0 = make_spd_matrix(n)

# Generate new point
m_new = riemannian_point_with_distance_and_angle(center, m0, dist=2.0, alpha_deg=45)

print("Distance from P:", distance_riemann(center, m_new))
angle,_ = mrtrf.angle_between_matrices(m0, m_new, center)
print("Angle (deg):", np.rad2deg(angle))



In [ ]:
# pathData = '/home/palatella/workspace/cybathlon_user/'

# ref20192020 = loadmat(f"{pathData}completeResults_20232024_angles_ref20192020.mat")

In [ ]:
############### CREATION OF user CVS  ###############

# pathData = '/home/palatella/workspace/cybathlon_user/'
# user1 = pd.read_csv(f"{pathData}completeResults_20192020.csv")   
# user1.rename(columns={'radius':'distance'}, inplace=True)

# btwClass_angles = loadmat(f"{pathData}completeResults_20192020_t_betwCl_angles.mat")['angles_to_otherClass']
# btwDistances = loadmat(f"{pathData}completeResults_20192020_btwWith_clss_distance.mat")
# anglesToBefore = loadmat(f"{pathData}completeResults_20192020_t_before_angles.mat")['angles_to_before']
# btwClass = btwDistances['btwClss']
# withClass = btwDistances['withClss']

# flatten_btwClass_angles = btwClass_angles.reshape(-1, 1, order='F')   
# user1['btw_angle'] = np.repeat(flatten_btwClass_angles, 2)  
# flatten_btwClass = btwClass.reshape(-1, 1, order='F')   
# user1['btw_dist'] = np.repeat(flatten_btwClass, 2)  
# user1['dts'] = 0
# user1['ovrRun'] = user1.runId
# user1['nSegment'] = 0
# user1.loc[user1.runId>=f1_1_winterStopX,'nSegment'] = 1
# user1['sameRef_angle'] = user1.angle
# user1['sameRef_distance'] = user1.distance
# user1['sameRef_btw_angle'] = user1.btw_angle

# user1['sameRef_bfr_angle'] = user1.btw_angle
# for idx, row in user1.iterrows():
#     user1.at[idx, 'sameRef_bfr_angle'] = anglesToBefore[row['band'],  int(row['runId']), row['task']]




# user2 = pd.read_csv(f"{pathData}completeResults_20232024.csv")
# user2.rename(columns={'radius':'distance'}, inplace=True)
# btwClass_angles = loadmat(f"{pathData}completeResults_20232024_t_betwCl_angles.mat")['angles_to_otherClass']
# btwDistances = loadmat(f"{pathData}completeResults_20232024_btwWith_clss_distance.mat")
# ref20192020 = loadmat(f"{pathData}completeResults_20232024_angles_ref20192020.mat")

# user2['sameRef_angle'] = np.zeros(user2.shape[0])
# user2['sameRef_distance'] = np.zeros(user2.shape[0])
# user2['sameRef_btw_angle'] = np.zeros(user2.shape[0])
# user2['sameRef_bfr_angle'] = np.zeros(user2.shape[0])

# for idx, row in user2.iterrows():
#     band = row['band']
#     runId = int(row['runId'])
#     task = row['task']
#     user2.at[idx, 'sameRef_angle'] = ref20192020['angles_to_zero'][band, runId, task]
#     user2.at[idx, 'sameRef_distance'] = ref20192020['d_ToEye'][band, runId, task]
#     user2.at[idx, 'sameRef_bfr_angle'] = ref20192020['angles_to_before'][band, runId, task]

# flatten_btwClass_angles = ref20192020['angles_to_otherClass'].reshape(-1, 1, order='F')   
# user2['sameRef_btw_angle'] = np.repeat(flatten_btwClass_angles, 2)  

# btwClass = btwDistances['btwClss']
# withClass = btwDistances['withClss']

# flatten_btwClass_angles = btwClass_angles.reshape(-1, 1, order='F')   
# user2['btw_angle'] = np.repeat(flatten_btwClass_angles, 2)  
# flatten_btwClass = btwClass.reshape(-1, 1, order='F')   
# user2['btw_dist'] = np.repeat(flatten_btwClass, 2)  
# user2['dts'] = 1
# user2['ovrRun'] = user2.runId + user1.ovrRun.max() + 1
# user2['nSegment'] = 2
# user2.loc[user2.runId>=f1_2_bhbf,'nSegment'] = 3


# user = pd.concat([user1, user2], ignore_index=True)


# winterRef = loadmat(f'{pathData}completeResults_20192020_centroidsMetrics_winterCenter.mat')
# user['wntr_distance'] = user.distance
# user['wntr_angle'] = user.angle
# idxWinter = np.array(range(f1_1_winterStopX, winterRef['d_ToEye'].shape[1]))
# for idx, row in user.iterrows():
#     if row['ovrRun'] in idxWinter:
#         user.at[idx, 'wntr_distance'] = winterRef['d_ToEye'][row['band'], int(row['ovrRun']), row['task']]
#         user.at[idx, 'wntr_angle'] = winterRef['angles_to_zero'][row['band'], int(row['ovrRun']), row['task']]
#         # print(f"Updated wntr metrics for ovrRun {idx, row['ovrRun']}")

# stats.add_cos_sin_angles(user, 'btw_angle')
# stats.add_cos_sin_angles(user, 'angle')
# stats.add_cos_sin_angles(user, 'sameRef_angle')
# stats.add_cos_sin_angles(user, 'sameRef_bfr_angle')
# stats.add_cos_sin_angles(user, 'sameRef_btw_angle')
# stats.add_cos_sin_angles(user, 'wntr_angle')


# # # saveName = 'completeResults_user_v2.csv'
# # # user.to_csv(f"{pathData}{saveName}", index=False)

# pathData = '/home/palatella/workspace/cybathlon_user/'
# user = pd.read_csv(f"{pathData}completeResults_user_v2.csv")


In [ ]:
use_single_row = False
task_range = [0] 
for t_idx, task in enumerate(task_range):
    print(t_idx, task)

0 0


In [ ]:
column_name = 'cos_angle'

f1_distance1 = stats.from_column_toNumpy_array(user, column_name)
f1_distance2 = stats.from_column_toNumpy_array(user2, column_name)


if column_name == 'radius':
    mergeTask = True
    averageOnTask = True
elif column_name == 'angle':
    mergeTask = False
    averageOnTask = False
else:
    mergeTask = False
    averageOnTask = False

# slopes = stats.plot_runs_grid(f1_distance1, f1_distance2, g6_distance1, g6_distance2, xlim=[0, 160], ylim=[0.5, 2.2], fitLine=True, fitSegments=fitSegments, mergeTask=True, averageOnTask=True)
